# Introduction to Deep Learning — Course Summary

**IBM AI Engineering Professional Certificate — Introduction to Deep Learning**

This notebook provides a **summary** of all lab materials from the *Introduction to Deep Learning* course (PyTorch fundamentals, linear regression, and logistic regression).

---
## Summary Table: Topics and Key Notebooks
---

| Topic | Notebooks | Main idea |
|-------|-----------|-----------|
| Datasets & transforms | 1_3_1_simple_data_set_v2, 1_3_2_datasets_and_transforms, 1_3_3_pre-built_datasets_and_transforms_v2 | Create datasets, apply transforms, use MNIST prebuilt |
| Linear regression 1D | 2_1prediction1dregression_v3, 2_2_linear_regression_one_parameter_v3, 2_3_training_slope_and_bias_v3 | Prediction, MSE cost, train slope/bias |
| SGD & PyTorch way | 3_1_stochastic_gradient_descent_v3, 3_3_pytorchway_v3, 3_6_training_and_validation_v3 | Minibatch SGD, nn.Module/Sequential, train/validation split |
| Multiple linear regression | 4_1_multiple_linear_regression_prediction_v2, 4_2_multiple_linear_regression_training_v2, 4_3_multi-target_linear_regression, 4_4_training_multiple_output_linear_regression | Multiple inputs/outputs, PyTorch built-ins |
| Logistic regression | 5_2_2bad_inshilization_logistic_regression_with_mean_square_error_v2_(1), 5_3_cross_entropy_logistic_regression_v2_(1) | Bad init & MSE vs cross-entropy |
| Neural networks | neural_network_for_breast_cancer_classification | Classification with a neural network |
| Final project | final_project_league_of_legends_match_predictor_(1) | League of Legends match prediction |

## Lab-by-Lab Summary (Objective, Strategy, Consolidated Code)
---

For each lab, the following format is used:
- **Objective:** Taken from the lab's Objective section.
- **Implementation Strategy:** How the code achieves the objective (data, model, training).
- **Consolidated Code Block:** Single runnable cell with imports, dataset, model, and training loop (no extra prints/plots).

### Lab: Simple Dataset (1_3_1_simple_data_set_v2)

**Objective:** Create a dataset in PyTorch and perform transformations on it.

**Implementation Strategy:**
- Define a custom `Dataset` subclass; implement `__len__` and `__getitem__`.
- Use `torchvision.transforms` (e.g. Compose) to apply transforms to data.
- Build a simple dataset (e.g. tensor data) and apply transforms for preprocessing.

In [1]:
# Consolidated: Simple dataset + transforms (1_3_1_simple_data_set_v2)
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
torch.manual_seed(0)

class SimpleDataset(Dataset):
    def __init__(self, transform=None):
        self.x = torch.arange(0, 10, 1.0).view(-1, 1)
        self.len = self.x.shape[0]
        self.transform = transform
    def __getitem__(self, index):
        x = self.x[index]
        if self.transform:
            x = self.transform(x)
        return x
    def __len__(self):
        return self.len

transform = transforms.Compose([transforms.Lambda(lambda t: t * 2)])
dataset = SimpleDataset(transform=transform)
sample = dataset[0]

### Lab: Datasets and Transforms (1_3_2_datasets_and_transforms)

**Objective:** Build an image dataset object and use Torchvision prebuilt transforms.

**Implementation Strategy:**
- Create or load an image dataset; use `Dataset` and optional `transform`.
- Apply Torchvision transforms (e.g. Resize, ToTensor, Normalize) via Compose.
- Use DataLoader for batching and iteration.

In [2]:
# Consolidated: Image dataset + Torchvision transforms (1_3_2_datasets_and_transforms)
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
torch.manual_seed(0)

class ImageDataset(Dataset):
    def __init__(self, n=50, transform=None):
        self.x = torch.randn(n, 1, 28, 28)
        self.y = torch.randint(0, 10, (n,))
        self.transform = transform
        self.len = n
    def __getitem__(self, i):
        x, y = self.x[i], self.y[i]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return self.len

transform = T.Compose([T.Normalize((0.5,), (0.5,))])
dataset = ImageDataset(n=50, transform=transform)
loader = DataLoader(dataset, batch_size=10, shuffle=True)
batch_x, batch_y = next(iter(loader))

### Lab: Pre-built Datasets and Transforms (1_3_3_pre-built_datasets_and_transforms_v2)

**Objective:** Use the MNIST prebuilt dataset in PyTorch.

**Implementation Strategy:**
- Load `torchvision.datasets.MNIST` with download=True and transform (e.g. ToTensor).
- Optionally apply Resize or other transforms; use DataLoader for training.

In [3]:
# Consolidated: MNIST pre-built dataset (1_3_3_pre-built_datasets_and_transforms_v2)
import torch
import torchvision.transforms as T
import torchvision.datasets as dsets
from torch.utils.data import DataLoader
torch.manual_seed(0)

train_dataset = dsets.MNIST(root='./data', train=True, download=True, transform=T.ToTensor())
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
x, y = next(iter(train_loader))

100.0%
100.0%
100.0%
100.0%


### Lab: Linear Regression 1D — Prediction (2_1prediction1dregression_v3)

**Objective:** Make predictions for multiple inputs; use the Linear class and build a custom module.

**Implementation Strategy:**
- Define a model using `nn.Linear` (or custom `nn.Module`) for 1D input and 1D output.
- Create synthetic 1D data; run forward passes to obtain predictions.

In [4]:
# Consolidated: Linear regression 1D prediction (2_1prediction1dregression_v3)
import torch
from torch.nn import Linear
torch.manual_seed(1)

class LR(torch.nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = Linear(input_size, output_size)
    def forward(self, x):
        return self.linear(x)

model = LR(1, 1)
x = torch.tensor([[1.0], [2.0], [3.0]])
yhat = model(x)

### Lab: Linear Regression 1D — Training One Parameter (2_2_linear_regression_one_parameter_v3)

**Objective:** Create a cost/criterion function using MSE (Mean Square Error).

**Implementation Strategy:**
- Use `nn.MSELoss()` or a manual MSE; train with a single learnable parameter (e.g. slope).
- Minimize cost with gradient descent (manual or optimizer).

In [5]:
# Consolidated: Linear regression 1D — train one parameter (2_2_linear_regression_one_parameter_v3)
import torch
torch.manual_seed(1)

def forward(x, w):
    return w * x
def criterion(yhat, y):
    return torch.mean((yhat - y) ** 2)

X = torch.tensor([[1.0], [2.0]])
Y = torch.tensor([[2.0], [4.0]])
w = torch.tensor(0.5, requires_grad=True)
optimizer = torch.optim.SGD([w], lr=0.1)
for _ in range(20):
    yhat = forward(X, w)
    loss = criterion(yhat, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

### Lab: Linear Regression 1D — Training Two Parameters (2_3_training_slope_and_bias_v3)

**Objective:** Train the model and visualize the loss results.

**Implementation Strategy:**
- Train slope and bias (two parameters); use MSE loss and an optimizer (e.g. SGD).
- Plot loss over iterations to monitor convergence.

In [6]:
# Consolidated: Linear regression 1D — train slope and bias (2_3_training_slope_and_bias_v3)
import torch
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-3, 3, 0.1).view(-1, 1)
        self.y = 1 * self.x - 1 + 0.1 * torch.randn(self.x.size())
        self.len = self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

def forward(x, w, b):
    return w * x + b
criterion = torch.nn.MSELoss()
w = torch.tensor(-15.0, requires_grad=True)
b = torch.tensor(-10.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.1)
loader = DataLoader(Data(), batch_size=1)
for epoch in range(10):
    for x, y in loader:
        yhat = forward(x, w, b)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: Stochastic Gradient Descent (3_1_stochastic_gradient_descent_v3)

**Objective:** Use SGD to train the model.

**Implementation Strategy:**
- Use `torch.optim.SGD`; update weights using minibatches (DataLoader) instead of full-batch gradient descent.
- Run a training loop: for each batch, forward, loss, backward, optimizer.step().

In [7]:
# Consolidated: SGD with DataLoader (3_1_stochastic_gradient_descent_v3)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-3, 3, 0.1).view(-1, 1)
        self.y = 1 * self.x - 1 + 0.1 * torch.randn(self.x.size())
        self.len = self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(Data(), batch_size=5)
for epoch in range(10):
    for x, y in loader:
        yhat = model(x)
        loss = criterion(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: PyTorch Way (3_3_pytorchway_v3)

**Objective:** Use PyTorch built-in functions to create a model.

**Implementation Strategy:**
- Build the model with `nn.Sequential` or `nn.Module` and `nn.Linear`.
- Use `nn.MSELoss()` and an optimizer; train with a standard loop.

In [8]:
# Consolidated: PyTorch way — Sequential + MSE (3_3_pytorchway_v3)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-3, 3, 0.1).view(-1, 1)
        self.y = 1 * self.x - 1 + 0.1 * torch.randn(self.x.size())
        self.len = self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(1, 1))
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(Data(), batch_size=10)
for epoch in range(10):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: Training and Validation Data (3_6_training_and_validation_v3)

**Objective:** Use the learning rate hyperparameter to improve model results.

**Implementation Strategy:**
- Split data into training and validation sets; train with different learning rates.
- Compare validation loss/accuracy to select a good learning rate and avoid overfitting.

In [ ]:
# Consolidated: Training and validation split (3_6_training_and_validation_v3)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T
import torchvision.datasets as dsets
torch.manual_seed(0)

dataset = dsets.MNIST(root='./data', train=True, download=True, transform=T.ToTensor())
train_ds, val_ds = random_split(dataset, [50000, 10000])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
for epoch in range(2):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
model.eval()
with torch.no_grad():
    correct = sum((model(x).argmax(1) == y).sum().item() for x, y in val_loader)
val_acc = correct / len(val_ds)

### Lab: Multiple Linear Regression — Prediction (4_1_multiple_linear_regression_prediction_v2)

**Objective:** Make predictions for multiple inputs; use Linear layers and build a custom module.

**Implementation Strategy:**
- Define a model with `nn.Linear` for multiple inputs (and optionally multiple outputs).
- Run forward passes on batched data.

In [ ]:
# Consolidated: Multiple linear regression — prediction (4_1_multiple_linear_regression_prediction_v2)
import torch
import torch.nn as nn
torch.manual_seed(1)

class LR(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        return self.linear(x)

model = LR(3, 2)
x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 3.0, 4.0]])
yhat = model(x)

### Lab: Multiple Linear Regression — Training (4_2_multiple_linear_regression_training_v2)

**Objective:** Create more complex models using PyTorch built-in functions.

**Implementation Strategy:**
- Use `nn.Sequential` or multiple `nn.Linear` layers; train with MSE and SGD (or Adam).
- Handle multiple features and optionally multiple outputs.

In [ ]:
# Consolidated: Multiple linear regression — training (4_2_multiple_linear_regression_training_v2)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.randn(100, 3)
        self.y = self.x @ torch.tensor([[1.0], [-0.5], [2.0]]) + 0.1 * torch.randn(100, 1)
        self.len = 100
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(3, 1))
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loader = DataLoader(Data(), batch_size=10)
for epoch in range(20):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: Multi-target Linear Regression (4_3_multi-target_linear_regression)

**Objective:** Make predictions using multiple samples (multi-output regression).

**Implementation Strategy:**
- Model outputs a vector per sample (e.g. `nn.Linear(in_features, out_features)` with out_features > 1).
- Use MSE or similar loss over all outputs; train with DataLoader.

In [ ]:
# Consolidated: Multi-target linear regression (4_3_multi-target_linear_regression)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.randn(80, 2)
        self.y = torch.randn(80, 3)
        self.len = 80
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Linear(2, 3)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loader = DataLoader(Data(), batch_size=8)
for epoch in range(15):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: Training Multiple Output Linear Regression (4_4_training_multiple_output_linear_regression)

**Objective:** Create complicated models using PyTorch built-in functions.

**Implementation Strategy:**
- Build a multi-output linear (or MLP) model; train with an optimizer and MSE loss.
- Use the same training loop pattern: forward, loss, backward, step.

In [ ]:
# Consolidated: Training multiple output linear regression (4_4_training_multiple_output_linear_regression)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(1)

class Data(Dataset):
    def __init__(self):
        self.x = torch.randn(100, 4)
        self.y = torch.randn(100, 2)
        self.len = 100
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loader = DataLoader(Data(), batch_size=10)
for epoch in range(20):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Lab: Bad Initialization & MSE for Logistic Regression (5_2_2bad_inshilization_logistic_regression_with_mean_square_error_v2_(1))

**Objective:** Understand how bad initialization can affect model accuracy (with MSE for logistic regression).

**Implementation Strategy:**
- Use a binary dataset, one `nn.Linear` with sigmoid, and **MSE** loss.
- Show that poor initialization (or flat MSE regions) can lead to poor convergence.

In [ ]:
# Consolidated: Logistic regression with MSE / bad init (5_2_2bad_inshilization_logistic_regression_with_mean_square_error_v2_(1))
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = (self.x[:, 0] > 0.2).float().unsqueeze(1)
        self.len = self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())
nn.init.constant_(model[0].weight, 0.5)
nn.init.constant_(model[0].bias, 0.0)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(Data(), batch_size=3)
for epoch in range(100):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
acc = (model(Data().x) > 0.5).float().eq(Data().y).float().mean().item()

### Lab: Cross-Entropy Logistic Regression (5_3_cross_entropy_logistic_regression_v2_(1))

**Objective:** See how cross-entropy with random initialization influences accuracy compared to MSE.

**Implementation Strategy:**
- Same setup as above but use **binary cross-entropy** (BCELoss or equivalent).
- Train with SGD; typically converges better than MSE for classification.

In [ ]:
# Consolidated: Cross-entropy logistic regression (5_3_cross_entropy_logistic_regression_v2_(1))
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)

class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-1, 1, 0.1).view(-1, 1)
        self.y = (self.x[:, 0] > 0.2).float().unsqueeze(1)
        self.len = self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]
    def __len__(self):
        return self.len

model = nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=2.0)
loader = DataLoader(Data(), batch_size=3)
for epoch in range(100):
    for x, y in loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
acc = (model(Data().x) > 0.5).float().eq(Data().y).float().mean().item()

### Lab: Neural Network for Breast Cancer Classification (neural_network_for_breast_cancer_classification)

**Objective:** Classify breast cancer (e.g. benign vs malignant) using a neural network.

**Implementation Strategy:**
- Load a tabular dataset (e.g. sklearn or CSV); build an MLP with hidden layers and sigmoid/softmax output.
- Use cross-entropy loss and an optimizer; split train/validation and report accuracy.

In [ ]:
# Consolidated: Neural network for breast cancer classification (neural_network_for_breast_cancer_classification)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0)

X = torch.randn(569, 30)
y = torch.randint(0, 2, (569,)).long()
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(
    nn.Linear(30, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 2)
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(10):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

### Final Project: League of Legends Match Predictor (final_project_league_of_legends_match_predictor_(1))

**Objective:** Apply course concepts to predict match outcomes (e.g. win/loss) using game-related features.

**Implementation Strategy:**
- Use a League of Legends dataset; preprocess features and labels.
- Build a classifier (logistic regression or neural network); train and evaluate (accuracy, etc.).

In [ ]:
# Consolidated: League of Legends match predictor — minimal (final_project_league_of_legends_match_predictor_(1))
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(0)

n_samples, n_features = 500, 20
X = torch.randn(n_samples, n_features)
y = torch.randint(0, 2, (n_samples,))
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = nn.Sequential(nn.Linear(n_features, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(5):
    for x, y in loader:
        y = y.unsqueeze(1).float()
        loss = criterion(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()